# Pascal VOC 2012 — data preparation + YOLO26 fine-tuning experiments

**Part 1** converts VOC XML to YOLO format and splits trainval 80/20.
**Part 2** runs the experiments; every run has its own cell and its own W&B run in project `voc-yolo26`.

# Part 1 — Data preparation

### Dataset check
Train/test overlap and whether the test XMLs contain boxes.

In [6]:
import os, glob
train = {os.path.basename(p) for p in glob.glob("../input/datasets/gopalbhattrai/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val/Annotations/*.xml")}
test_files = glob.glob("../input/datasets/gopalbhattrai/pascal-voc-2012-dataset/VOC2012_test/VOC2012_test/Annotations/*.xml")
test = {os.path.basename(p) for p in test_files}

# with_boxes = sum("<bndbox>" in open(p).read() for p in test_files)

# print("train:", len(train), "| test:", len(test), "| overlap:", len(train & test))
# print("test XMLs with boxes:", with_boxes, "of", len(test))

### Install Labelformat

In [ ]:
!pip install -q labelformat

### Paths and class names

In [3]:
import os
from pathlib import Path

ROOT = "/kaggle/input/datasets/gopalbhattrai/pascal-voc-2012-dataset"
TRAIN_PATH = Path(f"{ROOT}/VOC2012_train_val/VOC2012_train_val")
TEST_PATH  = Path(f"{ROOT}/VOC2012_test/VOC2012_test")
TRAIN_OUT  = Path("/kaggle/working/yolo_voc")
TEST_OUT   = Path("/kaggle/working/yolo_voc_test")

NAMES = ("aeroplane,bicycle,bird,boat,bottle,bus,car,cat,chair,cow,diningtable,"
         "dog,horse,motorbike,person,pottedplant,sheep,sofa,train,tvmonitor").split(",")
NAMES_STR = ",".join(NAMES)

def write_yaml(out_dir, **splits):
    lines = [f"path: {out_dir}"] + [f"{k}: {v}" for k, v in splits.items()]
    lines += [f"nc: {len(NAMES)}", "names:"] + [f"  {i}: {n}" for i, n in enumerate(NAMES)]
    (out_dir / "data.yaml").write_text("\n".join(lines) + "\n")

### Convert trainval XML → YOLO labels

In [4]:
!labelformat convert \
    --task object-detection \
    --input-format pascalvoc \
    --input-folder {TRAIN_PATH}/Annotations \
    --category-names {NAMES_STR} \
    --output-format yolov8 \
    --output-file {TRAIN_OUT}/data.yaml \
    --output-split train

if not (TRAIN_OUT / "images").exists():
    os.symlink(TRAIN_PATH / "JPEGImages", TRAIN_OUT / "images")

print("trainval label files:", len(list((TRAIN_OUT / "labels").glob("*.txt"))))
print(next((TRAIN_OUT / "labels").glob("*.txt")).read_text())

/bin/bash: line 1: labelformat: command not found


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/gopalbhattrai/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val/JPEGImages' -> '/kaggle/working/yolo_voc/images'

### Convert test XML → YOLO labels

In [ ]:
!labelformat convert \
    --task object-detection \
    --input-format pascalvoc \
    --input-folder {TEST_PATH}/Annotations \
    --category-names {NAMES_STR} \
    --output-format yolov8 \
    --output-file {TEST_OUT}/data.yaml \
    --output-split test

if not (TEST_OUT / "images").exists():
    os.symlink(TEST_PATH / "JPEGImages", TEST_OUT / "images")

test_ids = sorted(p.stem for p in (TEST_PATH / "Annotations").glob("*.xml"))
(TEST_OUT / "test.txt").write_text("\n".join(f"{TEST_OUT}/images/{i}.jpg" for i in test_ids))
write_yaml(TEST_OUT, train="test.txt", val="test.txt", test="test.txt")  # never train on this yaml
print("test:", len(test_ids), "images")

### Split trainval 80/20 + class balance check

In [ ]:
import random, collections

ids = sorted((TRAIN_PATH / "ImageSets/Main/trainval.txt").read_text().split())
random.Random(42).shuffle(ids)
cut = int(len(ids) * 0.8)
splits = {"train": ids[:cut], "val": ids[cut:]}

for name, split_ids in splits.items():
    (TRAIN_OUT / f"{name}.txt").write_text(
        "\n".join(f"{TRAIN_OUT}/images/{i}.jpg" for i in split_ids))
    print(f"{name}: {len(split_ids)} images")

write_yaml(TRAIN_OUT, train="train.txt", val="val.txt")

# balance check: share of each class's images that landed in val (should be near 20%)
def image_counts(id_list):
    c = collections.Counter()
    for i in id_list:
        lines = (TRAIN_OUT / "labels" / f"{i}.txt").read_text().splitlines()
        c.update({int(l.split()[0]) for l in lines if l.strip()})
    return c

tr, va = image_counts(splits["train"]), image_counts(splits["val"])
print(f"{'class':<12}{'train':>7}{'val':>7}{'val share':>11}")
for k, n in enumerate(NAMES):
    print(f"{n:<12}{tr[k]:>7}{va[k]:>7}{va[k] / (tr[k] + va[k]):>10.1%}")

### Visual check + class-mapping / difficult-object check

In [ ]:
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image

def read_labels(iid):
    return [l.split() for l in (TRAIN_OUT / "labels" / f"{iid}.txt").read_text().splitlines() if l.strip()]

# 1) draw 6 random train images with their YOLO boxes
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, iid in zip(axes.ravel(), random.sample(splits["train"], 6)):
    img = Image.open(TRAIN_OUT / "images" / f"{iid}.jpg"); W, H = img.size
    ax.imshow(img); ax.axis("off"); ax.set_title(iid)
    for c, cx, cy, w, h in read_labels(iid):
        cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
        ax.add_patch(patches.Rectangle((cx - w/2, cy - h/2), w, h, fill=False, color="lime", lw=2))
        ax.text(cx - w/2, cy - h/2, NAMES[int(c)], color="black", backgroundcolor="lime", fontsize=8)
plt.tight_layout(); plt.show()

# 2) compare one image's YOLO labels with its original XML
for iid in splits["train"]:
    objs = ET.parse(TRAIN_PATH / "Annotations" / f"{iid}.xml").getroot().findall("object")
    if any(o.findtext("difficult") == "1" for o in objs):
        break

xml_all  = sorted(o.findtext("name").strip() for o in objs)
xml_easy = sorted(o.findtext("name").strip() for o in objs if o.findtext("difficult") != "1")
yolo     = sorted(NAMES[int(l[0])] for l in read_labels(iid))

print("image:", iid)
print("XML, all objects:   ", xml_all)
print("XML, non-difficult: ", xml_easy)
print("YOLO label file:    ", yolo)
print("→", "difficult objects KEPT" if yolo == xml_all
      else "difficult objects DROPPED" if yolo == xml_easy
      else "MISMATCH: class mapping is wrong")

# Part 2 — Experiments

## Install

In [ ]:
!pip install -q -U ultralytics wandb

## Imports

In [ ]:
import random
import time
from pathlib import Path

import pandas as pd
import torch
import yaml
import wandb
from ultralytics import YOLO, settings

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## W&B login

In [ ]:
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("wandb"))

settings.update({"wandb": True})  # Ultralytics now logs every epoch into the active W&B run

## Data
Same folders as Part 1. Re-read here so Part 2 still works after a kernel restart (files in /kaggle/working survive it, variables don't).

In [ ]:
TRAIN_OUT = Path("/kaggle/working/yolo_voc")
DATA = TRAIN_OUT / "data.yaml"                                # train.txt (9,232) + val.txt (2,308)
TEST_DATA = Path("/kaggle/working/yolo_voc_test/data.yaml")   # test set: used ONLY in the last section
RUNS = "/kaggle/working/runs"                                 # local folder for weights, results.csv, plots
PROJECT = "voc-yolo26"                                        # W&B project

NAMES = list(yaml.safe_load(DATA.read_text())["names"].values())
train_images = (TRAIN_OUT / "train.txt").read_text().split()
val_images = (TRAIN_OUT / "val.txt").read_text().split()
print(len(NAMES), "classes |", len(train_images), "train images |", len(val_images), "val images")

### Overfit subset
32 fixed train images. The sanity checks train **and** validate on these same images.

In [ ]:
random.seed(0)
overfit_images = random.sample(train_images, 32)
Path("/kaggle/working/overfit.txt").write_text("\n".join(overfit_images))

OVERFIT_DATA = Path("/kaggle/working/overfit.yaml")
OVERFIT_DATA.write_text(yaml.safe_dump({
    "path": str(TRAIN_OUT),
    "train": "/kaggle/working/overfit.txt",
    "val": "/kaggle/working/overfit.txt",       # same images on purpose
    "names": dict(enumerate(NAMES)),
}))

### Train-eval subset
1,000 fixed train images. After every run, the best model is scored on them and on val → train-vs-val gap.

In [ ]:
random.seed(1)
train_eval_images = random.sample(train_images, 1000)
Path("/kaggle/working/train_eval.txt").write_text("\n".join(train_eval_images))

TRAIN_EVAL_DATA = Path("/kaggle/working/train_eval.yaml")
TRAIN_EVAL_DATA.write_text(yaml.safe_dump({
    "path": str(TRAIN_OUT),
    "train": str(TRAIN_OUT / "train.txt"),       # required key, not used for scoring
    "val": "/kaggle/working/train_eval.txt",    # these are the images that get scored
    "names": dict(enumerate(NAMES)),
}))

### COCO → VOC class names
14 classes have the same name in both datasets. These 6 are spelled differently.

In [ ]:
ALIASES = {
    "airplane": "aeroplane",
    "motorcycle": "motorbike",
    "couch": "sofa",
    "potted plant": "pottedplant",
    "dining table": "diningtable",
    "tv": "tvmonitor",
}

## Config

In [ ]:
SEED = 0
EPOCHS = 20          # placeholder: set after the pilot
BATCH = 32
IMGSZ = 640
LR = 0.001           # AdamW learning rate
PATIENCE = 10        # stop if val mAP doesn't improve for 10 epochs
WARMUP_EPOCHS = 1.0
CLOSE_MOSAIC = 3     # last 3 epochs without mosaic (default 10 is too many for short runs)

## Helper functions
Only model setup and logging. Training itself is written out in every run cell.

In [ ]:
def apply_coco_aliases(model):
    """Rename 6 COCO classes to their VOC spelling, so Ultralytics copies their classifier rows too."""
    model.model.names = {i: ALIASES.get(name, name) for i, name in model.model.names.items()}
    matched = len(set(NAMES) & set(model.model.names.values()))
    print(f"Classifier rows that will be copied from COCO: {matched}/{len(NAMES)}")
    return matched

In [ ]:
def add_epoch_timer(model):
    """Log train seconds and val seconds of every epoch to W&B. Call AFTER wandb.init()."""
    wandb.run.define_metric("time/*", step_metric="epoch")
    times = {"train_sec": [], "val_sec": []}

    def epoch_start(trainer):
        times["start"] = time.time()

    def train_done(trainer):
        times["train_done"] = time.time()

    def epoch_done(trainer):
        train_sec = times["train_done"] - times["start"]
        val_sec = time.time() - times["train_done"]
        times["train_sec"].append(train_sec)
        times["val_sec"].append(val_sec)
        wandb.log({"time/train_sec": train_sec, "time/val_sec": val_sec, "epoch": trainer.epoch + 1}, commit=False)

    model.add_callback("on_train_epoch_start", epoch_start)
    model.add_callback("on_train_epoch_end", train_done)
    model.add_callback("on_fit_epoch_end", epoch_done)
    return times

In [ ]:
def log_fit_summary(model, times, epochs):
    """Read results.csv of the finished run (one row per epoch) and log how training went."""
    df = pd.read_csv(model.trainer.csv)
    df.columns = df.columns.str.strip()
    map95 = df["metrics/mAP50-95(B)"]

    summary = {
        "time/mean_train_sec_per_epoch": sum(times["train_sec"]) / len(times["train_sec"]),
        "time/mean_val_sec_per_epoch": sum(times["val_sec"]) / len(times["val_sec"]),
        "fit/epochs_run": len(df),
        "fit/stopped_early": len(df) < epochs,
        "fit/best_epoch": int(map95.idxmax()) + 1,
        "fit/peak_val_mAP50-95": float(map95.max()),
        "fit/last_val_mAP50-95": float(map95.iloc[-1]),
        "fit/drop_from_peak": float(map95.max() - map95.iloc[-1]),   # > 0: val got worse at the end → overfitting hint
    }
    wandb.run.summary.update(summary)
    print(f"epochs run {len(df)} | best epoch {summary['fit/best_epoch']} | "
          f"peak val mAP50-95 {summary['fit/peak_val_mAP50-95']:.3f} | drop from peak {summary['fit/drop_from_peak']:.3f}")
    return df

In [ ]:
def evaluate(weights, data, split):
    """Score a model on one split. Returns (metrics dict, per-class W&B table)."""
    m = YOLO(str(weights)).val(data=str(data), split=split, batch=BATCH, imgsz=IMGSZ,
                               plots=False, verbose=False, project=RUNS, name="eval", exist_ok=True)
    metrics = {"mAP50": m.box.map50, "mAP50-95": m.box.map, "precision": m.box.mp, "recall": m.box.mr}
    rows = m.summary()                      # one dict per class
    columns = list(rows[0].keys())
    table = wandb.Table(columns=columns, data=[[row[c] for c in columns] for row in rows])
    return metrics, table

In [ ]:
def log_val_and_gap(weights):
    """Score the best model on val and on the 1,000 train images; log both and the gap between them."""
    val_m, val_table = evaluate(weights, DATA, "val")
    train_m, _ = evaluate(weights, TRAIN_EVAL_DATA, "val")
    gap = train_m["mAP50-95"] - val_m["mAP50-95"]

    wandb.run.summary.update({f"final/val_{k}": v for k, v in val_m.items()})
    wandb.run.summary.update({f"final/train_{k}": v for k, v in train_m.items()})
    wandb.run.summary["final/gap_mAP50-95"] = gap
    wandb.log({"per_class/val": val_table})
    print(f"val mAP50-95 {val_m['mAP50-95']:.3f} | train mAP50-95 {train_m['mAP50-95']:.3f} | gap {gap:+.3f}")

## Test run — time one epoch
1 epoch on 10% of train. Gives the minutes per full epoch (to set `EPOCHS`) and checks that W&B logging works.

In [ ]:
model = YOLO("yolo26n.pt")
apply_coco_aliases(model)

wandb.init(project=PROJECT, name="Pilot | YOLO26n | 10% data, 1 epoch", group="pilot",
           config={"purpose": "time one epoch + test logging", "fraction": 0.1})
times = add_epoch_timer(model)

model.train(data=str(DATA), epochs=1, fraction=0.1, batch=BATCH, imgsz=IMGSZ,
            optimizer="AdamW", lr0=LR, seed=SEED,
            close_mosaic=0,                  # keep mosaic on, like in the real runs
            plots=False, project=RUNS, name="pilot", exist_ok=True)

full_epoch_sec = times["train_sec"][0] / 0.1 + times["val_sec"][0]   # train part scales with data size, val doesn't
wandb.run.summary["pilot/est_full_epoch_min"] = full_epoch_sec / 60
print(f"Estimated full epoch: {full_epoch_sec / 60:.1f} min | 10 epochs ≈ {10 * full_epoch_sec / 3600:.1f} h"
      f" | 20 epochs ≈ {20 * full_epoch_sec / 3600:.1f} h")
wandb.finish()

## Run 0 — Zero-shot COCO baseline (no training)
VOC val labels are rewritten to COCO class IDs, then the plain COCO model is scored on them.
Only the 20 VOC classes are scored: a bus predicted as "truck" counts as a missed bus, but the truck box is not counted as a false positive.

In [ ]:
coco_names = YOLO("yolo26n.pt").names                           # {0: 'person', 1: 'bicycle', ...}
coco_id = {name: i for i, name in coco_names.items()}
voc_to_coco_name = {voc: coco for coco, voc in ALIASES.items()}

ZS = Path("/kaggle/working/yolo_voc_zeroshot")
(ZS / "labels").mkdir(parents=True, exist_ok=True)
if not (ZS / "images").exists():
    (ZS / "images").symlink_to(TRAIN_OUT / "images")

zs_images = []
for img in val_images:
    stem = Path(img).stem
    new_lines = []
    for line in (TRAIN_OUT / "labels" / f"{stem}.txt").read_text().splitlines():
        if line.strip():
            c, cx, cy, w, h = line.split()
            voc_name = NAMES[int(c)]
            coco_class = coco_id[voc_to_coco_name.get(voc_name, voc_name)]
            new_lines.append(f"{coco_class} {cx} {cy} {w} {h}")
    (ZS / "labels" / f"{stem}.txt").write_text("\n".join(new_lines))
    zs_images.append(str(ZS / "images" / f"{stem}.jpg"))

(ZS / "val.txt").write_text("\n".join(zs_images))
ZS_DATA = ZS / "data.yaml"
ZS_DATA.write_text(yaml.safe_dump({
    "path": str(ZS), "train": str(ZS / "val.txt"), "val": str(ZS / "val.txt"), "names": coco_names,
}))
print(len(zs_images), "val images relabelled with COCO IDs")

In [ ]:
wandb.init(project=PROJECT, name="R0 | YOLO26n zero-shot (COCO, no training)", group="baseline",
           config={"run": "R0", "architecture": "yolo26n", "pretrained": "COCO", "training": "none"})

metrics, table = evaluate("yolo26n.pt", ZS_DATA, "val")
wandb.run.summary.update({f"final/val_{k}": v for k, v in metrics.items()})
wandb.log({"per_class/val": table})
print(metrics)
wandb.finish()

### Sanity check — base config
Train and validate on the same 32 images, augmentation off, 100 epochs. `batch=nbs=16` so every batch is an optimizer step.
PASS if peak mAP50 ≥ 0.9: this setup can learn at all.

In [ ]:
model = YOLO("yolo26n.pt")
apply_coco_aliases(model)

wandb.init(project=PROJECT, name="Sanity | overfit 32 imgs | YOLO26n base", group="sanity",
           config={"purpose": "overfit 32 images", "architecture": "yolo26n", "optimizer": "AdamW", "lr0": LR})
times = add_epoch_timer(model)

model.train(data=str(OVERFIT_DATA), epochs=100, patience=0, batch=16, nbs=16, imgsz=IMGSZ,
            seed=SEED, warmup_epochs=WARMUP_EPOCHS,
            optimizer="AdamW", lr0=LR,
            hsv_h=0, hsv_s=0, hsv_v=0, translate=0, scale=0, fliplr=0, mosaic=0, close_mosaic=0,
            plots=False, project=RUNS, name="sanity_base", exist_ok=True)

df = log_fit_summary(model, times, epochs=100)
peak = df["metrics/mAP50(B)"].max()
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()


## Run 1 — Base
Full fine-tune, AdamW `lr0=0.001`, Ultralytics default augmentation, all 20 classifier rows copied from COCO.

In [ ]:
model = YOLO("yolo26n.pt")
head_rows = apply_coco_aliases(model)

wandb.init(project=PROJECT, name="R1 | YOLO26n base", group="ablation", config={
    "run": "R1",
    "classifier_rows_from_coco": f"{head_rows}/20",
    "architecture": "yolo26n",
    "freeze": "none",
    "optimizer": "AdamW", "lr0": LR,
    "augmentation": "Ultralytics default (hsv, translate 0.1, scale 0.5, fliplr 0.5, mosaic)",
    "class_weighting": "none",
    "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42",
})
times = add_epoch_timer(model)

model.train(data=str(DATA), epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
            optimizer="AdamW", lr0=LR,
            project=RUNS, name="r1_base", exist_ok=True)

log_fit_summary(model, times, epochs=EPOCHS)
log_val_and_gap(model.trainer.best)
wandb.finish()

## Run 2 — No classifier transfer
Same as Run 1, but `cls_remap=False` and no aliases: the 20 classifier output rows start **random** instead of copied from COCO. Everything else still comes from COCO.

In [ ]:
model = YOLO("yolo26n.pt")
head_rows = 0                    # no aliases, and cls_remap=False below: all 20 rows start random

wandb.init(project=PROJECT, name="R2 | YOLO26n + random classifier rows", group="ablation", config={
    "run": "R2",
    "classifier_rows_from_coco": "0/20 (random init)",
    "architecture": "yolo26n",
    "freeze": "none",
    "optimizer": "AdamW", "lr0": LR,
    "augmentation": "Ultralytics default (hsv, translate 0.1, scale 0.5, fliplr 0.5, mosaic)",
    "class_weighting": "none",
    "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42",
})
times = add_epoch_timer(model)

model.train(data=str(DATA), epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
            optimizer="AdamW", lr0=LR,
            cls_remap=False,
            project=RUNS, name="r2_no_head_transfer", exist_ok=True)

log_fit_summary(model, times, epochs=EPOCHS)
log_val_and_gap(model.trainer.best)
wandb.finish()

### Sanity check — frozen backbone
Train and validate on the same 32 images, augmentation off, 100 epochs. `batch=nbs=16` so every batch is an optimizer step.
PASS if peak mAP50 ≥ 0.9: this setup can learn at all.

In [ ]:
model = YOLO("yolo26n.pt")
apply_coco_aliases(model)

wandb.init(project=PROJECT, name="Sanity | overfit 32 imgs | YOLO26n frozen backbone", group="sanity",
           config={"purpose": "overfit 32 images", "architecture": "yolo26n", "freeze": "layers 0-10 (backbone)"})
times = add_epoch_timer(model)

model.train(data=str(OVERFIT_DATA), epochs=100, patience=0, batch=16, nbs=16, imgsz=IMGSZ,
            seed=SEED, warmup_epochs=WARMUP_EPOCHS,
            optimizer="AdamW", lr0=LR, freeze=11,
            hsv_h=0, hsv_s=0, hsv_v=0, translate=0, scale=0, fliplr=0, mosaic=0, close_mosaic=0,
            plots=False, project=RUNS, name="sanity_freeze_backbone", exist_ok=True)

df = log_fit_summary(model, times, epochs=100)
peak = df["metrics/mAP50(B)"].max()
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()

## Run 3 — Frozen backbone
`freeze=11`: layers 0–10 (the whole backbone) never update. Neck and head train.

In [ ]:
model = YOLO("yolo26n.pt")
head_rows = apply_coco_aliases(model)

wandb.init(project=PROJECT, name="R3 | YOLO26n + frozen backbone (rerun)", group="ablation", config={
    "run": "R3",
    "classifier_rows_from_coco": f"{head_rows}/20",
    "architecture": "yolo26n",
    "freeze": "layers 0-10 (backbone)",
    "optimizer": "AdamW", "lr0": LR,
    "augmentation": "Ultralytics default (hsv, translate 0.1, scale 0.5, fliplr 0.5, mosaic)",
    "class_weighting": "none",
    "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42",
})
times = add_epoch_timer(model)

model.train(data=str(DATA), epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
            optimizer="AdamW", lr0=LR,
            freeze=11,
            project=RUNS, name="r3_freeze_backbone_rerun", exist_ok=True,cache=True)

log_fit_summary(model, times, epochs=EPOCHS)
log_val_and_gap(model.trainer.best)
wandb.finish()

### Sanity check — head only
Train and validate on the same 32 images, augmentation off, 100 epochs. `batch=nbs=16` so every batch is an optimizer step.
PASS if peak mAP50 ≥ 0.9: this setup can learn at all.

In [ ]:
model = YOLO("yolo26n.pt")
apply_coco_aliases(model)

wandb.init(project=PROJECT, name="Sanity | overfit 32 imgs | YOLO26n head only", group="sanity",
           config={"purpose": "overfit 32 images", "architecture": "yolo26n", "freeze": "layers 0-22 (head trains)"})
times = add_epoch_timer(model)

model.train(data=str(OVERFIT_DATA), epochs=100, patience=0, batch=16, nbs=16, imgsz=IMGSZ,
            seed=SEED, warmup_epochs=WARMUP_EPOCHS,
            optimizer="AdamW", lr0=LR, freeze=23,
            hsv_h=0, hsv_s=0, hsv_v=0, translate=0, scale=0, fliplr=0, mosaic=0, close_mosaic=0,
            plots=False, project=RUNS, name="sanity_head_only", exist_ok=True,cache=True)

df = log_fit_summary(model, times, epochs=100)
peak = df["metrics/mAP50(B)"].max()
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()

## Run 4 — Head only
`freeze=23`: layers 0–22 frozen, only the detection head trains. Expected to underfit.

In [ ]:
model = YOLO("yolo26n.pt")
head_rows = apply_coco_aliases(model)

wandb.init(project=PROJECT, name="R4 | YOLO26n + head only", group="ablation", config={
    "run": "R4",
    "classifier_rows_from_coco": f"{head_rows}/20",
    "architecture": "yolo26n",
    "freeze": "layers 0-22 (head only)",
    "optimizer": "AdamW", "lr0": LR,
    "augmentation": "Ultralytics default (hsv, translate 0.1, scale 0.5, fliplr 0.5, mosaic)",
    "class_weighting": "none",
    "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42",
})
times = add_epoch_timer(model)

model.train(data=str(DATA), epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
            optimizer="AdamW", lr0=LR,
            freeze=23,
            project=RUNS, name="r4_head_only", exist_ok=True)

log_fit_summary(model, times, epochs=EPOCHS)
log_val_and_gap(model.trainer.best)
wandb.finish()

## Run 5 — No augmentation
Every augmentation set to 0: colour (hsv), translate, scale, horizontal flip, mosaic. Expected to overfit more.

In [ ]:
model = YOLO("yolo26n.pt")
head_rows = apply_coco_aliases(model)

wandb.init(project=PROJECT, name="R5 | YOLO26n + no augmentation", group="ablation", config={
    "run": "R5",
    "classifier_rows_from_coco": f"{head_rows}/20",
    "architecture": "yolo26n",
    "freeze": "none",
    "optimizer": "AdamW", "lr0": LR,
    "augmentation": "none",
    "class_weighting": "none",
    "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42",
})
times = add_epoch_timer(model)

model.train(data=str(DATA), epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
            optimizer="AdamW", lr0=LR,
            hsv_h=0, hsv_s=0, hsv_v=0, translate=0, scale=0, fliplr=0, mosaic=0,
            project=RUNS, name="r5_no_aug", exist_ok=True)

log_fit_summary(model, times, epochs=EPOCHS)
log_val_and_gap(model.trainer.best)
wandb.finish()

## Run 6 — Heavy augmentation
Default augmentation plus: `scale` 0.5→0.9, `translate` 0.1→0.2, `mixup` 0→0.2, `cutmix` 0→0.2.

In [ ]:
model = YOLO("yolo26n.pt")
head_rows = apply_coco_aliases(model)

wandb.init(project=PROJECT, name="R6 | YOLO26n + heavy augmentation", group="ablation", config={
    "run": "R6",
    "classifier_rows_from_coco": f"{head_rows}/20",
    "architecture": "yolo26n",
    "freeze": "none",
    "optimizer": "AdamW", "lr0": LR,
    "augmentation": "default + scale 0.9, translate 0.2, mixup 0.2, cutmix 0.2",
    "class_weighting": "none",
    "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42",
})
times = add_epoch_timer(model)

model.train(data=str(DATA), epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
            optimizer="AdamW", lr0=LR,
            scale=0.9, translate=0.2, mixup=0.2, cutmix=0.2,
            project=RUNS, name="r6_heavy_aug", exist_ok=True)

log_fit_summary(model, times, epochs=EPOCHS)
log_val_and_gap(model.trainer.best)
wandb.finish()

## Run 7 — Class weighting
`cls_pw=0.5`: loss weight ∝ (1/class frequency)^0.5, so sheep counts ≈3.8× person. Check per-class AP of sheep and cow.

In [ ]:
model = YOLO("yolo26n.pt")
head_rows = apply_coco_aliases(model)

wandb.init(project=PROJECT, name="R7 | YOLO26n + class weighting 0.5", group="ablation", config={
    "run": "R7",
    "classifier_rows_from_coco": f"{head_rows}/20",
    "architecture": "yolo26n",
    "freeze": "none",
    "optimizer": "AdamW", "lr0": LR,
    "augmentation": "Ultralytics default (hsv, translate 0.1, scale 0.5, fliplr 0.5, mosaic)",
    "class_weighting": "inverse frequency, power 0.5",
    "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42",
})
times = add_epoch_timer(model)

model.train(data=str(DATA), epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
            optimizer="AdamW", lr0=LR,
            cls_pw=0.5,
            project=RUNS, name="r7_class_weight", exist_ok=True)

log_fit_summary(model, times, epochs=EPOCHS)
log_val_and_gap(model.trainer.best)
wandb.finish()

### Sanity check — SGD
Train and validate on the same 32 images, augmentation off, 100 epochs. `batch=nbs=16` so every batch is an optimizer step.
PASS if peak mAP50 ≥ 0.9: this setup can learn at all.

In [ ]:
model = YOLO("yolo26n.pt")
apply_coco_aliases(model)

wandb.init(project=PROJECT, name="Sanity | overfit 32 imgs | YOLO26n SGD", group="sanity",
           config={"purpose": "overfit 32 images", "architecture": "yolo26n", "optimizer": "SGD", "lr0": 0.01})
times = add_epoch_timer(model)

model.train(data=str(OVERFIT_DATA), epochs=100, patience=0, batch=16, nbs=16, imgsz=IMGSZ,
            seed=SEED, warmup_epochs=WARMUP_EPOCHS,
            optimizer="SGD", lr0=0.01, momentum=0.937,
            hsv_h=0, hsv_s=0, hsv_v=0, translate=0, scale=0, fliplr=0, mosaic=0, close_mosaic=0,
            plots=False, project=RUNS, name="sanity_sgd", exist_ok=True)

df = log_fit_summary(model, times, epochs=100)
peak = df["metrics/mAP50(B)"].max()
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()

## Run 8 — SGD optimizer
SGD with momentum 0.937 and `lr0=0.01`: SGD needs a ~10× larger learning rate than AdamW.

In [ ]:
model = YOLO("yolo26n.pt")
head_rows = apply_coco_aliases(model)

wandb.init(project=PROJECT, name="R8 | YOLO26n + SGD", group="ablation", config={
    "run": "R8",
    "classifier_rows_from_coco": f"{head_rows}/20",
    "architecture": "yolo26n",
    "freeze": "none",
    "optimizer": "SGD", "lr0": 0.01, "momentum": 0.937,
    "augmentation": "Ultralytics default (hsv, translate 0.1, scale 0.5, fliplr 0.5, mosaic)",
    "class_weighting": "none",
    "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42",
})
times = add_epoch_timer(model)

model.train(data=str(DATA), epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
            optimizer="SGD", lr0=0.01, momentum=0.937,
            project=RUNS, name="r8_sgd", exist_ok=True)

log_fit_summary(model, times, epochs=EPOCHS)
log_val_and_gap(model.trainer.best)
wandb.finish()

### Sanity check — YOLO26s
Train and validate on the same 32 images, augmentation off, 100 epochs. `batch=nbs=16` so every batch is an optimizer step.
PASS if peak mAP50 ≥ 0.9: this setup can learn at all.

In [ ]:
model = YOLO("yolo26s.pt")
apply_coco_aliases(model)

wandb.init(project=PROJECT, name="Sanity | overfit 32 imgs | YOLO26s base", group="sanity",
           config={"purpose": "overfit 32 images", "architecture": "yolo26s", "optimizer": "AdamW", "lr0": LR})
times = add_epoch_timer(model)

model.train(data=str(OVERFIT_DATA), epochs=100, patience=0, batch=16, nbs=16, imgsz=IMGSZ,
            seed=SEED, warmup_epochs=WARMUP_EPOCHS,
            optimizer="AdamW", lr0=LR,
            hsv_h=0, hsv_s=0, hsv_v=0, translate=0, scale=0, fliplr=0, mosaic=0, close_mosaic=0,
            plots=False, project=RUNS, name="sanity_yolo26s", exist_ok=True)

df = log_fit_summary(model, times, epochs=100)
peak = df["metrics/mAP50(B)"].max()
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()

## Run 9 — Bigger model
`yolo26s` instead of `yolo26n`, everything else like Run 1. If it runs out of GPU memory, lower `batch`: thanks to `nbs=64` the optimizer still steps every ~64 images.

In [ ]:
model = YOLO("yolo26s.pt")
head_rows = apply_coco_aliases(model)

wandb.init(project=PROJECT, name="R9 | YOLO26s base", group="ablation", config={
    "run": "R9",
    "classifier_rows_from_coco": f"{head_rows}/20",
    "architecture": "yolo26s",
    "freeze": "none",
    "optimizer": "AdamW", "lr0": LR,
    "augmentation": "Ultralytics default (hsv, translate 0.1, scale 0.5, fliplr 0.5, mosaic)",
    "class_weighting": "none",
    "epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42",
})
times = add_epoch_timer(model)

model.train(data=str(DATA), epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
            optimizer="AdamW", lr0=LR,
            project=RUNS, name="r9_yolo26s", exist_ok=True)

log_fit_summary(model, times, epochs=EPOCHS)
log_val_and_gap(model.trainer.best)
wandb.finish()

## Final test evaluation — run ONCE, at the very end
Pick the model(s) to report. Copy the run ID from the W&B run page URL and the run's folder name.

In [ ]:
RUN_ID = "w7l3pjhz"                              # from the W&B run URL
WEIGHTS = f"/kaggle/working/runs/r3_freeze_backbone/weights/best.pt"          # folder of that run

metrics, table = evaluate(WEIGHTS, TEST_DATA, "test")

wandb.init(project=PROJECT, id=RUN_ID, resume="must")   # add test results to the same W&B run
wandb.run.summary.update({f"test/{k}": v for k, v in metrics.items()})
wandb.log({"per_class/test": table})
wandb.finish()
print(metrics) 